<a href="https://colab.research.google.com/github/AileenLavelle/PBC_Object_Detection/blob/main/Boca_nomask_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import json
import numpy as np

!pip install sahi ultralytics
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

In [4]:
# YOLOv11 model
# yolo11n.pt (fastest), yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt (most accurate)
model_path = "yolo11n.pt"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=model_path,
    confidence_threshold=0.3,
    device="cuda:0"
)

In [9]:
import os
import json
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Load YOLOv11 model
model = YOLO("yolo11n.pt")

# Setup paths
image_folder = "/content/Boca_Raton"
output_folder = "/content/Boca_Raton/output_yolo11"
os.makedirs(output_folder, exist_ok=True)
ground_truth_json = "/content/Boca_Raton/_annotations.coco.json"

with open(ground_truth_json) as f:
    gt = json.load(f)
# Re-generate predictions with correct category ID
predictions = []

for img_data in tqdm(gt['images'], desc="Processing images"):
    img_path = os.path.join(image_folder, img_data['file_name'])

    if not os.path.exists(img_path):
        continue

    # Change device from 'cuda:0' to 'cpu' as no GPU is available.
    results = model(img_path, conf=0.3, device='cpu', verbose=False)

    for result in results:
        boxes = result.boxes
        for i in range(len(boxes)):
            class_id = int(boxes.cls[i])
            # YOLO class 8 = boat in COCO, map to your category ID 1
            if class_id == 8:
                xyxy = boxes.xyxy[i].cpu().numpy()
                conf = float(boxes.conf[i])
                bbox = [float(xyxy[0]), float(xyxy[1]),
                        float(xyxy[2] - xyxy[0]), float(xyxy[3] - xyxy[1])]

                predictions.append({
                    "image_id": img_data['id'],
                    "category_id": 1,  # Changed from 8 to 1
                    "bbox": bbox,
                    "score": conf
                })

# Save and evaluate
pred_json = os.path.join(output_folder, "predictions.json")
with open(pred_json, 'w') as f:
    json.dump(predictions, f)

coco_gt = COCO(ground_truth_json)
coco_dt = coco_gt.loadRes(predictions) if predictions else coco_gt.loadRes([])

coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
coco_eval.params.catIds = [1]  # Evaluate category ID 1 (Boat)
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

print(f"\nTotal predictions: {len(predictions)}")
print(f"mAP@0.5:0.95: {coco_eval.stats[0]:.3f}")
print(f"mAP@0.5: {coco_eval.stats[1]:.3f}")

Processing images: 100%|██████████| 46/46 [00:10<00:00,  4.42it/s]

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.032
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.064
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.032
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.016
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.305
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.036
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.042
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [11]:
!pip install sahi
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

# Setup SAHI model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolo11n.pt",
    confidence_threshold=0.3,
    device="cuda:0"
)

predictions_sahi = []

for img_data in tqdm(gt['images'], desc="Processing with SAHI"):
    img_path = os.path.join(image_folder, img_data['file_name'])
    if not os.path.exists(img_path):
        continue

    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2
    )

    for obj in result.object_prediction_list:
        if obj.category.id == 8:  # YOLO boat class
            bbox = obj.bbox
            predictions_sahi.append({
                "image_id": img_data['id'],
                "category_id": 1,  # Your boat category
                "bbox": [bbox.minx, bbox.miny, bbox.maxx - bbox.minx, bbox.maxy - bbox.miny],
                "score": obj.score.value
            })

# Evaluate SAHI results
coco_dt_sahi = coco_gt.loadRes(predictions_sahi)
coco_eval_sahi = COCOeval(coco_gt, coco_dt_sahi, 'bbox')
coco_eval_sahi.params.catIds = [1]
coco_eval_sahi.evaluate()
coco_eval_sahi.accumulate()
coco_eval_sahi.summarize()

print(f"\nSAHI Total predictions: {len(predictions_sahi)} (baseline: 36)")



# 1. Check what's being missed - visualize false negatives
import cv2
from PIL import Image

# Get ground truth boats
gt_boats = [ann for ann in gt['annotations'] if ann['category_id'] == 1]
pred_image_ids = set(p['image_id'] for p in predictions)
gt_image_ids = set(ann['image_id'] for ann in gt_boats)

print(f"Images with GT boats: {len(gt_image_ids)}")
print(f"Images with predictions: {len(pred_image_ids)}")
print(f"Images with boats but no predictions: {len(gt_image_ids - pred_image_ids)}")
print(f"\nTotal GT boats: {len(gt_boats)}")
print(f"Total predictions: {len(predictions)}")

Processing with SAHI:   0%|          | 0/46 [00:00<?, ?it/s]

Performing prediction on 1 slices.


Processing with SAHI:   2%|▏         | 1/46 [00:00<00:12,  3.52it/s]

Performing prediction on 1 slices.


Processing with SAHI:   4%|▍         | 2/46 [00:00<00:11,  3.77it/s]

...

Performing prediction on 1 slices.


Processing with SAHI:  98%|█████████▊| 45/46 [00:11<00:00,  4.21it/s]

Performing prediction on 1 slices.


Processing with SAHI: 100%|██████████| 46/46 [00:11<00:00,  3.99it/s]

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.032
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.064
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.033
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.016
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.312
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.036
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.042
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.042
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10